# Candidate Sampling and Popularity-Based Recommendation Baseline

**DS5500 Data Science Capstone — Behavior-Aware Sequential Recommendation for Next Purchase Prediction**

This notebook implements a popularity-based recommendation baseline and a reproducible candidate sampling pipeline using the Synerise RecSys 2025 dataset.

The pipeline uses purchase sequences and temporal evaluation splits generated in `Step1_SequenceConstruction.ipynb`. It constructs shared candidate sets, ranks products by training purchase frequency, and evaluates next-purchase recommendations using HitRate@10, NDCG@10, and Mean Reciprocal Rank (MRR).

The saved candidate sets and shared evaluation utilities support consistent comparisons with purchase-only SASRec and behavior-aware SASRec models.

In [1]:

import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from evaluation_utils import (
    evaluate_model_predictions,
    evaluate_model_dataset
)

# File paths
ARTIFACT_DIR = Path("artifacts")
SEQUENCES_PATH = ARTIFACT_DIR / "sequences_and_splits.pkl"

# Load shared sequences and evaluation splits
with open(SEQUENCES_PATH, "rb") as f:
    data = pickle.load(f)

# Shared datasets
train_sequences = data["train_sequences"]
val_targets = data["val_targets"]
test_targets = data["test_targets"]
val_eval_users = data["val_eval_users"]
test_eval_users = data["test_eval_users"]

# Shared item vocabulary
item_to_idx = data["item_to_idx"]
idx_to_item = data["idx_to_item"]
all_items = data["all_items"]
n_items = data["n_items"]

# Evaluation configuration
config = data["config"]

MAX_SEQ_LEN = config["MAX_SEQ_LEN"]
VAL_DAYS = config["VAL_DAYS"]
TEST_DAYS = config["TEST_DAYS"]

VAL_CUTOFF = config["val_cutoff"]
TEST_CUTOFF = config["test_cutoff"]

N_NEGATIVES = 100
EVAL_K = 10
RANDOM_SEED = 42

print("Shared data loaded successfully.")
print(f"Training users: {len(train_sequences):,}")
print(f"Validation users: {len(val_eval_users):,}")
print(f"Test users: {len(test_eval_users):,}")
print(f"Vocabulary size: {n_items:,}")
print(f"Negative candidates per user: {N_NEGATIVES}")
print(f"Validation cutoff: {VAL_CUTOFF}")
print(f"Test cutoff: {TEST_CUTOFF}")


/Users/sonikapuvvada/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/sonikapuvvada/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Shared data loaded successfully.
Training users: 246,307
Validation users: 34,127
Test users: 35,637
Vocabulary size: 449,958
Negative candidates per user: 100
Validation cutoff: 2022-11-10 00:09:15
Test cutoff: 2022-11-24 00:09:15


## 2. Data Overview and Coverage

We use the purchase sequences, evaluation targets, and temporal splits generated by `Step1_SequenceConstruction.ipynb`. The training data contains purchase histories for 246,307 users, with 34,127 validation users and 35,637 test users eligible for next-purchase evaluation.

The shared vocabulary contains 449,958 products, represented by original product IDs and mapped to encoded indices for sequential recommendation models. Index 0 is reserved for padding.

We examine item coverage in the training sequences and the proportion of evaluation targets previously observed during training. Products without training purchases have no observed popularity, making this distinction particularly relevant to the purchase-frequency baseline.

In [2]:

# Training item coverage
train_items = {
    item
    for sequence in train_sequences.values()
    for item in sequence
}

all_items_set = set(all_items)

print("Training Item Coverage")
print("-" * 40)
print(f"Total vocabulary items: {len(all_items_set):,}")
print(f"Items seen in training: {len(train_items):,}")
print(f"Items unseen in training: {len(all_items_set - train_items):,}")
print(f"Coverage: {len(train_items) / len(all_items_set) * 100:.2f}%")

# Evaluation target coverage
coverage_rows = []

for split_name, eval_users, targets in [
    ("Validation", val_eval_users, val_targets),
    ("Test", test_eval_users, test_targets)
]:
    seen = sum(targets[user] in train_items for user in eval_users)
    unseen = len(eval_users) - seen

    coverage_rows.append({
        "Split": split_name,
        "Users": len(eval_users),
        "Seen targets": seen,
        "Unseen targets": unseen,
        "Seen target %": round(seen / len(eval_users) * 100, 2)
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

# Verify item vocabulary compatibility
assert len(all_items_set) == n_items
assert all(item in item_to_idx for item in all_items_set)

print("\nVocabulary consistency checks passed.")


Training Item Coverage
----------------------------------------
Total vocabulary items: 449,958
Items seen in training: 356,014
Items unseen in training: 93,944
Coverage: 79.12%


,Split,Users,Seen targets,Unseen targets,Seen target %
0,Validation,34127,20790,13337,60.92
1,Test,35637,17927,17710,50.30



Vocabulary consistency checks passed.


## 3. Candidate Sampling

We construct reproducible candidate sets for next-purchase evaluation. Each set contains one positive item (the user's actual next purchase) and 100 unique negative items sampled from the full shared item vocabulary.

Candidate sampling uses a fixed random seed to support reproducibility. The resulting candidate sets are shared across the popularity baseline, purchase-only SASRec, and behavior-aware SASRec to ensure consistent model comparisons.

### 3.1 Candidate Generation

We use NumPy-based sampling to efficiently generate candidate products without replacement, excluding the positive target from the negative samples. Candidate products are retained as original product IDs and can be converted to encoded indices using the shared `item_to_idx` mapping when needed.

In [3]:

# Shared candidate vocabulary
item_pool = np.asarray(all_items)

def sample_candidates(target_item, item_pool, rng, n_negatives=100):
    """Sample negative products and include the positive target."""
    if target_item not in item_to_idx:
        raise ValueError("Target is not in the shared vocabulary.")

    if n_negatives < 0 or n_negatives >= len(item_pool):
        raise ValueError("Invalid number of negatives.")

    selected_indices = rng.choice(
        len(item_pool),
        size=n_negatives,
        replace=False
    )

    negatives = item_pool[selected_indices]

    # Replace the positive target if it was sampled as a negative
    while target_item in negatives:
        target_positions = np.where(negatives == target_item)[0]

        for position in target_positions:
            replacement = rng.choice(item_pool)

            while replacement == target_item or replacement in negatives:
                replacement = rng.choice(item_pool)

            negatives[position] = replacement

    candidates = np.append(negatives, target_item)
    rng.shuffle(candidates)

    return candidates.tolist()


def generate_candidate_sets(eval_users, targets, item_pool,
                            n_negatives=100, seed=42):
    """Generate reproducible candidate sets for an evaluation split."""
    rng = np.random.default_rng(seed)

    return {
        user: sample_candidates(
            targets[user], item_pool, rng, n_negatives
        )
        for user in eval_users
    }


### 3.2 Generate and Validate Evaluation Candidates

We generate candidate sets for all eligible validation and test users using the shared item vocabulary. For each user, we sample 100 unique negative products without replacement, exclude the actual next-purchase target from the negative samples, and randomly shuffle the resulting 101 candidates.

We use a fixed random seed of 42 to ensure reproducibility. Validation and test candidate sets are generated separately using the same sampling procedure.

We then verify that every candidate set contains exactly 101 unique products, includes the true target exactly once, and contains only products from the shared vocabulary.

In [4]:

# Generate shared candidate sets
val_candidate_sets = generate_candidate_sets(
    val_eval_users,
    val_targets,
    item_pool,
    n_negatives=N_NEGATIVES,
    seed=RANDOM_SEED
)

test_candidate_sets = generate_candidate_sets(
    test_eval_users,
    test_targets,
    item_pool,
    n_negatives=N_NEGATIVES,
    seed=RANDOM_SEED
)

print(f"Validation candidate sets: {len(val_candidate_sets):,}")
print(f"Test candidate sets: {len(test_candidate_sets):,}")
print(f"Candidates per user: {N_NEGATIVES + 1}")


Validation candidate sets: 34,127
Test candidate sets: 35,637
Candidates per user: 101


In [5]:

def validate_candidate_sets(candidate_sets, eval_users, targets):
    """Verify candidate counts, targets, uniqueness, and vocabulary."""
    assert set(candidate_sets) == set(eval_users)

    for user, candidates in candidate_sets.items():
        assert len(candidates) == N_NEGATIVES + 1
        assert len(set(candidates)) == len(candidates)
        assert candidates.count(targets[user]) == 1
        assert all(item in item_to_idx for item in candidates)

    return len(candidate_sets)


for name, candidates, users, targets in [
    ("Validation", val_candidate_sets, val_eval_users, val_targets),
    ("Test", test_candidate_sets, test_eval_users, test_targets)
]:
    n_valid = validate_candidate_sets(candidates, users, targets)
    print(f"{name}: {n_valid:,} candidate sets passed all checks")


Validation: 34,127 candidate sets passed all checks
Test: 35,637 candidate sets passed all checks


### 3.3 Verify Candidate Sampling Reproducibility

We compare the newly generated validation and test candidate sets against the previously saved shared evaluation artifact.

This verifies that consolidating the candidate sampling implementation has not changed the sampled products or their ordering.

Identical candidate sets ensure that model comparisons remain reproducible and that the popularity baseline and sequential recommendation models can evaluate the same products.

In [6]:

saved_path = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"

if saved_path.exists():
    with open(saved_path, "rb") as f:
        saved_artifacts = pickle.load(f)

    val_match = (
        val_candidate_sets == saved_artifacts["val_candidate_sets"]
    )
    test_match = (
        test_candidate_sets == saved_artifacts["test_candidate_sets"]
    )

    print("Validation candidates match:", val_match)
    print("Test candidates match:", test_match)

    assert val_match and test_match, (
        "Regenerated candidates differ from saved artifact."
    )
else:
    print("No previous candidate artifact found.")
    print("New candidate sets will be saved in Section 6.")


Validation candidates match: True
Test candidates match: True


## 4. Popularity-Based Recommendation Baseline

We implement a non-personalized popularity baseline that ranks products according to their purchase frequencies in the training data.

For each product, its popularity score is the total number of purchase events across all training sequences. Repeated purchases are counted separately, while products without training purchases receive a score of zero.

The baseline uses only training purchase histories to prevent validation or test information from influencing its recommendations. It provides a simple benchmark for evaluating whether sequential recommendation models improve next-purchase prediction beyond global product popularity.

### 4.1 Compute Popularity Scores

We calculate purchase frequencies and define functions for scoring and ranking candidate products. Candidates are ranked in descending order of popularity, with ascending product IDs used to break ties deterministically.

In [7]:

# Count training purchase events per product
item_popularity = Counter(
    item
    for sequence in train_sequences.values()
    for item in sequence
)

def score_candidates_popularity(candidates, item_popularity):
    """Assign training purchase counts to candidate products."""
    return [
        item_popularity.get(item, 0)
        for item in candidates
    ]


def rank_candidates_popularity(candidates, item_popularity):
    """Rank products by popularity, breaking ties by product ID."""
    return sorted(
        candidates,
        key=lambda item: (-item_popularity.get(item, 0), item)
    )


print(f"Products with training purchases: {len(item_popularity):,}")

print("\nTop 10 most purchased products:")
for item, count in item_popularity.most_common(10):
    print(f"Product {item}: {count:,} purchases")


Products with training purchases: 356,014

Top 10 most purchased products:
Product 52586: 971 purchases
Product 219232: 890 purchases
Product 301108: 690 purchases
Product 1243077: 673 purchases
Product 234831: 554 purchases
Product 1492640: 521 purchases
Product 867128: 489 purchases
Product 1202337: 481 purchases
Product 1086081: 462 purchases
Product 940412: 427 purchases


## 5. Popularity Baseline Evaluation

We evaluate the popularity baseline using the shared validation candidate sets, with 100 sampled negative products and one actual next-purchase target per user.

For each user, candidate products receive scores based on their training purchase frequencies. The shared `evaluation_utils.py` module ranks candidates and calculates HitRate@10, NDCG@10, and Mean Reciprocal Rank (MRR).

This standardized evaluation approach allows direct comparison with purchase-only and behavior-aware SASRec models once their predictions are available.

### 5.1 Full Validation Results

We evaluate all eligible validation users using the shared candidate sets and report aggregate recommendation performance.

In [8]:

# Calculate popularity scores for all validation users
val_model_scores = {
    user: score_candidates_popularity(
        candidates, item_popularity
    )
    for user, candidates in val_candidate_sets.items()
}

# Evaluate using the shared evaluation module
val_summary, val_results = evaluate_model_dataset(
    candidate_sets=val_candidate_sets,
    model_scores=val_model_scores,
    targets=val_targets,
    k=EVAL_K
)

print("Popularity Baseline — Validation Results")
print("-" * 45)

display(pd.DataFrame([val_summary]))


Popularity Baseline — Validation Results
---------------------------------------------


,n_users,HitRate@10,NDCG@10,MRR
0,34127,0.388226,0.256578,0.228811


### 5.2 Performance by Training Item Coverage

We analyze popularity baseline performance separately for products observed during training and products with no recorded training purchases.

This distinction is important because purchase-frequency recommendations cannot distinguish unseen products based on historical purchase counts. In contrast, behavior-aware models may have additional information from cart interactions.

We compare HitRate@10, NDCG@10, and MRR across both groups to identify limitations of the popularity baseline and motivate the potential value of incorporating cart behavior.

In [9]:

# Identify whether each target appeared in training
val_results["target_seen_in_train"] = (
    val_results["target_item"].isin(train_items)
)

# Aggregate evaluation metrics by target coverage
coverage_results = (
    val_results
    .groupby("target_seen_in_train")
    .agg(
        n_users=("user_id", "count"),
        hit_rate_10=("HitRate@10", "mean"),
        ndcg_10=("NDCG@10", "mean"),
        mrr=("RR", "mean")
    )
    .reset_index()
)

coverage_results["target_group"] = coverage_results[
    "target_seen_in_train"
].map({
    True: "Seen in training",
    False: "Unseen in training"
})

display(
    coverage_results[
        ["target_group", "n_users", "hit_rate_10",
         "ndcg_10", "mrr"]
    ]
)


,target_group,n_users,hit_rate_10,ndcg_10,mrr
0,Unseen in training,13337,0.000000,0.000000,0.011105
1,Seen in training,20790,0.637278,0.421176,0.368471


**Interpretation.** The popularity baseline performs substantially better on products that appeared in training. Products without training purchases receive a popularity score of zero, preventing them from outranking candidates with positive purchase counts.

In the validation evaluation, unseen targets consistently rank outside the top 10 because every unseen target is competing against at least 61 candidates with positive training popularity. Consequently, HitRate@10 and NDCG@10 are zero for this group.

This result highlights a limitation of purchase-frequency recommendations and motivates investigating whether cart interactions improve predictions for products with limited purchase history. However, any advantage of the behavior-aware model must be established through direct evaluation using the same users and candidates.

## 6. Save Evaluation Artifacts and Results

We save the shared candidate sets and popularity baseline validation results to support reproducibility and consistent comparisons across recommendation models.

The shared candidate artifact contains the validation and test candidate sets along with their sampling configuration. We also save the aggregate and per-user popularity baseline validation metrics as CSV files.

All models will use identical evaluation users, targets, and candidate sets, with shared ranking metrics implemented in `evaluation_utils.py`. The test set is reserved for final evaluation after model development and selection.

### 6.1 Save Shared Candidates and Validation Metrics

Evaluation artifacts are saved in the `artifacts/` directory for reuse by the purchase-only SASRec and behavior-aware SASRec pipelines.

In [10]:

# Ensure the output directory exists
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Package shared candidate sets and configuration
evaluation_artifacts = {
    "val_candidate_sets": val_candidate_sets,
    "test_candidate_sets": test_candidate_sets,
    "n_negatives": N_NEGATIVES,
    "eval_k": EVAL_K,
    "random_seed": RANDOM_SEED,
    "candidate_pool": "full_item_vocabulary",
    "include_unseen_targets": True,
    "max_seq_len": MAX_SEQ_LEN,
    "val_cutoff": VAL_CUTOFF,
    "test_cutoff": TEST_CUTOFF
}

# Save shared candidate sets
candidates_path = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"

with open(candidates_path, "wb") as f:
    pickle.dump(
        evaluation_artifacts,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

# Save aggregate validation metrics
summary_path = ARTIFACT_DIR / "popularity_val_summary.csv"
pd.DataFrame([val_summary]).to_csv(summary_path, index=False)

# Save per-user validation results
results_path = ARTIFACT_DIR / "popularity_val_user_results.csv"
val_results.to_csv(results_path, index=False)

print("Saved files:")
for path in [candidates_path, summary_path, results_path]:
    print(f"- {path} ({path.stat().st_size / 1e6:.2f} MB)")


Saved files:
- artifacts/shared_evaluation_candidates.pkl (35.29 MB)
- artifacts/popularity_val_summary.csv (0.00 MB)
- artifacts/popularity_val_user_results.csv (1.55 MB)


### 6.2 Verify Saved Artifacts

We reload the saved candidate sets and validation results to verify that the files were written correctly.

We confirm that the saved candidates match the original generated sets and that the saved aggregate metrics reproduce the evaluation results.

These checks ensure that the artifacts are consistent and ready to be reused by other recommendation models.

In [11]:

# Reload shared candidate sets
with open(candidates_path, "rb") as f:
    saved_artifacts = pickle.load(f)

# Reload validation results
saved_summary = pd.read_csv(summary_path)
saved_user_results = pd.read_csv(results_path)

# Verify saved candidates
assert saved_artifacts["val_candidate_sets"] == val_candidate_sets
assert saved_artifacts["test_candidate_sets"] == test_candidate_sets

# Verify saved metrics
for metric in ["HitRate@10", "NDCG@10", "MRR"]:
    assert np.isclose(
        saved_summary.loc[0, metric],
        val_summary[metric]
    )

# Verify number of evaluation users
assert len(saved_user_results) == len(val_eval_users)

print("Saved candidate sets verified.")
print("Saved validation metrics verified.")
print("Saved per-user results verified.")
print("\nAll artifact checks passed.")


Saved candidate sets verified.
Saved validation metrics verified.
Saved per-user results verified.

All artifact checks passed.


## 7. Integration with Sequential Recommendation Models

The shared candidate sets and evaluation utilities support consistent comparisons between the popularity baseline, purchase-only SASRec, and behavior-aware SASRec.

**Shared evaluation inputs:**

- `artifacts/sequences_and_splits.pkl`: Evaluation users, next-purchase targets, and shared item vocabulary.
- `artifacts/shared_evaluation_candidates.pkl`: Validation and test candidate sets containing 100 negative products and one positive target per user.
- `evaluation_utils.py`: Reusable functions for calculating HitRate@10, NDCG@10, and MRR.

**Model integration:** Each recommendation model should score the saved candidates for every eligible evaluation user, preserving candidate order. Candidate products are stored as original product IDs and should be converted using the shared `item_to_idx` mapping when required by the model.

Prediction scores can then be passed to `evaluate_model_dataset()` to calculate aggregate and per-user ranking metrics. All models should use identical evaluation users, targets, and candidates for the primary comparison.

**Evaluation considerations:** The reported metrics are based on sampled candidate sets rather than full-catalog ranking. Products without training purchases are retained in the evaluation. The test set is reserved for final evaluation after model development and selection.

The cart-aware model must also use the agreed temporal cutoffs and avoid incorporating interactions that occur after the relevant prediction time.

In [12]:

# Example: Evaluate another recommendation model
#
# 1. Load the shared candidate sets and targets.
# 2. Convert candidate IDs to model indices if needed.
# 3. Generate prediction scores aligned with candidate order.
# 4. Evaluate using the shared module.
#
# Example call once model scores are available:
#
# model_summary, model_results = evaluate_model_dataset(
#     candidate_sets=val_candidate_sets,
#     model_scores=model_scores,
#     targets=val_targets,
#     k=EVAL_K
# )
